![MuJoCo banner](https://raw.githubusercontent.com/google-deepmind/mujoco/main/banner.png)







# Import in apple verion
### Check Critical Package Version
✅ JAX: 0.6.2
✅ MuJoCo: 3.3.6
✅ Brax: 0.13.0
✅ Flax: 0.10.7

In [1]:
from apple_mujoco_setup import *

Failed to import warp: No module named 'warp'
Failed to import mujoco.mjx.third_party.mujoco_warp as mujoco_warp: No module named 'warp'
Detected macOS: arm64
Using MUJOCO_GL=glfw for macOS
Forcing JAX to run on CPU backend
Mujoco installation and rendering backend OK
JAX: 0.6.2
MuJoCo: 3.3.6
Brax: 0.13.0
Flax: 0.10.7
Installing mediapy & ffmpeg if needed...

Checking media packages...
✓ ffmpeg available
✓ mediapy available
[CpuDevice(id=0), CpuDevice(id=1), CpuDevice(id=2), CpuDevice(id=3), CpuDevice(id=4), CpuDevice(id=5), CpuDevice(id=6), CpuDevice(id=7)]
JAX device count: 8


In [2]:
registry.manipulation.ALL_ENVS

('AlohaHandOver',
 'AlohaSinglePegInsertion',
 'PandaPickCube',
 'UR10PickCube',
 'PandaPickCubeOrientation',
 'PandaPickCubeCartesian',
 'PandaOpenCabinet',
 'PandaRobotiqPushCube',
 'LeapCubeReorient',
 'LeapCubeRotateZAxis')

# UR10e with Hand E - Run Diagnostics to xml
## Torque model

Let's start off with the simplest environment, simply picking up a cube with the Franka Emika Panda.

In [5]:
env_name = 'UR10PickCube'
env = registry.load(env_name)
env_cfg = registry.get_default_config(env_name)
print("Loaded:", env)
print("Action size:", env.action_size)
print("Gripper site ID:", env._gripper_site)

m = env.mj_model

print("\nBodies:", [mujoco.mj_id2name(m, mujoco.mjtObj.mjOBJ_BODY, i) for i in range(m.nbody)])
print("Joints:", [mujoco.mj_id2name(m, mujoco.mjtObj.mjOBJ_JOINT, i) for i in range(m.njnt)])
print("Sites:", [mujoco.mj_id2name(m, mujoco.mjtObj.mjOBJ_SITE, i) for i in range(m.nsite)])
print(env.action_size)
print([env.mj_model.joint(i).name for i in range(env.mj_model.njnt)])

✓ Using keyframe: 'task_home'
  Initial qpos size: 15
  Robot joints: [-1.5708 -1.5708  1.5708 -1.5708 -1.5708  0.      0.      0.    ]
Loaded: <mujoco_playground._src.manipulation.my_ur10.ur10pick.UR10PickCube object at 0x12c1aef50>
Action size: 7
Gripper site ID: 1

Bodies: ['world', 'base', 'shoulder_link', 'upper_arm_link', 'forearm_link', 'wrist_1_link', 'wrist_2_link', 'wrist_3_link', 'robotiq_hande_mount', 'hande_left_finger', 'hande_right_finger', 'box', 'mocap_target']
Joints: ['shoulder_pan_joint', 'shoulder_lift_joint', 'elbow_joint', 'wrist_1_joint', 'wrist_2_joint', 'wrist_3_joint', 'hande_left_finger_joint', 'hande_right_finger_joint', None]
Sites: ['attachment_site', 'tcp', 'left_finger_touch_site', 'right_finger_touch_site']
7
['shoulder_pan_joint', 'shoulder_lift_joint', 'elbow_joint', 'wrist_1_joint', 'wrist_2_joint', 'wrist_3_joint', 'hande_left_finger_joint', 'hande_right_finger_joint', '']


/Users/matthiasweiss/miniconda3/envs/mujoco/lib/python3.10/site-packages/mujoco/mjx/_src/mesh.py:141: UserWarning: Mesh "coupler" has a coplanar face with more than 20 vertices. This may lead to performance issues and inaccuracies in collision detection. Consider decimating the mesh.
  warnings.warn(
/Users/matthiasweiss/miniconda3/envs/mujoco/lib/python3.10/site-packages/mujoco/mjx/_src/mesh.py:141: UserWarning: Mesh "hande" has a coplanar face with more than 20 vertices. This may lead to performance issues and inaccuracies in collision detection. Consider decimating the mesh.
  warnings.warn(


In [6]:
print("Actuators:", env.mj_model.nu)
print([env.mj_model.actuator(i).name for i in range(env.mj_model.nu)])

Actuators: 7
['shoulder_pan', 'shoulder_lift', 'elbow', 'wrist_1', 'wrist_2', 'wrist_3', 'hande_fingers_actuator']


## Rollout before training


In [ ]:
# '../mujoco_playground/_src/manipulation/my_ur10/xmls/mjx_scene.xml'
# '../mujoco_playground/_src/manipulation/my_ur10/xmls/test_grvity.xml'
# '/../mujoco_playground/external_deps/mujoco_menagerie/franka_emika_panda/scene.xml'
# '../../mujoco_playground/_src/manipulation/my_ur10/universal_robots_ur10e/ur10e.xml'

## Gravity tests and rollouts

Ball falling to the floor

In [ ]:
model = mujoco.MjModel.from_xml_path('../../mujoco_playground/_src/manipulation/my_ur10/xmls/test_grvity.xml')
data = mujoco.MjData(model)

mujoco.viewer.launch(model, data)

Now load the torque model alone to check if the robot is gravity controlled. It needs to be the torque model. because position model allway goes to the desired position

In [ ]:
model = mujoco.MjModel.from_xml_path("../../mujoco_playground/_src/manipulation/my_ur10/xmls/mjx_scene_torque.xml")
data = mujoco.MjData(model)

mujoco.viewer.launch(model, data)

## Test Movements of Actuators


In [ ]:
model = mujoco.MjModel.from_xml_path("../../mujoco_playground/_src/manipulation/my_ur10/xmls/mjx_scene_torque.xml")
data = mujoco.MjData(model)

mujoco.viewer.launch(model, data)

## Train Policy

Let's train the pick cube policy and visualize rollouts. The policy takes roughly 3 minutes to train on an RTX 4090.

In [ ]:
from mujoco_playground.config import manipulation_params
ppo_params = manipulation_params.brax_ppo_config(env_name)
ppo_params

In [ ]:
from copy import deepcopy

# Make a copy so you don't overwrite the original defaults
fast_ppo_params = deepcopy(ppo_params)

# --- Speedup adjustments ---
fast_ppo_params["num_timesteps"] = 10_000_000 
# fast_ppo_params["learning_rate"] = 0.002        
# fast_ppo_params["episode_length"] = 200        
# fast_ppo_params["unroll_length"] = 10         

# (Optional) reduce parallel envs to lower compute requirements
# fast_ppo_params["num_envs"] = 2048              # from 8192 or 4096 or 2048
# print("Adjusted num_envs:", fast_ppo_params["num_envs"])

# (Optional) tweak batch sizes accordingly !! Needs to be 
# fast_ppo_params["batch_size"] = 256             
# fast_ppo_params["num_minibatches"] = 16         

ppo_params = fast_ppo_params
ppo_params

### PPO

In [ ]:
x_data, y_data, y_dataerr = [], [], []
times = [datetime.now()]


def progress(num_steps, metrics):
  clear_output(wait=True)

  times.append(datetime.now())
  x_data.append(num_steps)
  y_data.append(metrics["eval/episode_reward"])
  y_dataerr.append(metrics["eval/episode_reward_std"])

  plt.xlim([0, ppo_params["num_timesteps"] * 1.25])
  plt.xlabel("# environment steps")
  plt.ylabel("reward per episode")
  plt.title(f"y={y_data[-1]:.3f}")
  plt.errorbar(x_data, y_data, yerr=y_dataerr, color="blue")

  display(plt.gcf())

ppo_training_params = dict(ppo_params)
network_factory = ppo_networks.make_ppo_networks
if "network_factory" in ppo_params:
  del ppo_training_params["network_factory"]
  network_factory = functools.partial(
      ppo_networks.make_ppo_networks,
      **ppo_params.network_factory
  )

train_fn = functools.partial(
    ppo.train, **dict(ppo_training_params),
    network_factory=network_factory,
    progress_fn=progress,
    seed=1
)

In [ ]:
make_inference_fn, params, metrics = train_fn(
    environment=env,
    wrap_env_fn=wrapper.wrap_for_brax_training,
)
if y_data:
    import numpy as np
    best_idx = int(np.argmax(y_data))

### Metrics of Training
Rewards over 1000 yield decent results
With default training time reward is 1347 after 20'152'320

In [ ]:
print(f"time to jit: {times[1] - times[0]}")
print(f"time to train: {times[-1] - times[1]}")
print(f"Highest Reward: {y_data[best_idx]:.3f} ± {y_dataerr[best_idx]:.3f} at step {x_data[best_idx]}")

## Visualize Rollouts

In [ ]:
jit_reset = jax.jit(env.reset)
jit_step = jax.jit(env.step)
jit_inference_fn = jax.jit(make_inference_fn(params, deterministic=True))

In [ ]:
rng = jax.random.PRNGKey(42)
rollout = []
n_episodes = 1

for _ in range(n_episodes):
  state = jit_reset(rng)
  rollout.append(state)
  for i in range(env_cfg.episode_length):
    act_rng, rng = jax.random.split(rng)
    ctrl, _ = jit_inference_fn(state.obs, act_rng)
    state = jit_step(state, ctrl)
    rollout.append(state)

render_every = 1
frames = env.render(rollout[::render_every])
rewards = [s.reward for s in rollout]
media.show_video(frames, fps=1.0 / env.dt / render_every)

While the above policy is very simple, the work was extended using the Madrona batch renderer, and policies were transferred on a real robot. We encourage folks to check out the Madrona-MJX tutorial notebooks ([part 1](https://colab.research.google.com/github/google-deepmind/mujoco_playground/blob/main/learning/notebooks/training_vision_1.ipynb) and [part 2](https://colab.research.google.com/github/google-deepmind/mujoco_playground/blob/main/learning/notebooks/training_vision_2.ipynb))!